# 02 · Calibration
#
**Question:** How well calibrated are Bet365's implied probabilities?
#
We compare what the (closing) price implied vs what actually occurred for
H, D and A. We do **not** call any discrepancy an edge — that is explored
later, and only out-of-sample.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd

from src import data as D
from src import probabilities as P
from src import calibration as C
from src import plotting as plt

primary = P.add_probability_columns(D.load_processed())
print("Primary rows:", len(primary))


Primary rows: 2280


## Calibration by outcome (probability buckets)
#
Each bucket groups matches by the normalised implied probability assigned to
that outcome. `implied` = mean bucket probability; `observed` = realised
frequency; ci_low/ci_high = Wilson binomial interval.

In [2]:
for o in ("H", "D", "A"):
    tbl = C.probability_buckets(primary, o)
    print(f"\n==============================================")
    print(f"  {o} calibration")
    print("==============================================")
    print(C.calibration_statistics(tbl))
    print(tbl[["bucket", "n", "implied", "observed", "ci_low", "ci_high", "diff"]].round(4).to_string())



  H calibration
{'buckets': 8, 'total_matches': 2269, 'mae': 0.032531549715679696, 'bias': 0.009863991047517765}
      bucket    n  implied  observed  ci_low  ci_high    diff
0  0.10-0.20  179   0.1605    0.1620  0.1152   0.2230  0.0015
1  0.20-0.30  330   0.2568    0.2000  0.1604   0.2465 -0.0568
2  0.30-0.40  482   0.3547    0.3423  0.3014   0.3858 -0.0123
3  0.40-0.50  481   0.4519    0.4699  0.4257   0.5145  0.0179
4  0.50-0.60  373   0.5497    0.5764  0.5257   0.6255  0.0267
5  0.60-0.70  242   0.6519    0.7521  0.6940   0.8022  0.1002
6  0.70-0.80  141   0.7447    0.7801  0.7049   0.8406  0.0355
7  0.80-0.90   41   0.8299    0.8049  0.6599   0.8977 -0.0250

  D calibration
{'buckets': 3, 'total_matches': 2272, 'mae': 0.00871451490556375, 'bias': 0.002887540202626471}
      bucket     n  implied  observed  ci_low  ci_high    diff
0  0.10-0.20   299   0.1660    0.1438  0.1086   0.1881 -0.0221
1  0.20-0.30  1445   0.2639    0.2692  0.2470   0.2927  0.0053
2  0.30-0.40   528   0.317

### H calibration curve

In [3]:
fig = plt.calibration_curve(C.probability_buckets(primary, "H"), title="H: implied vs observed")
fig.show()


In [4]:
fig = plt.calibration_curve(C.probability_buckets(primary, "D"), title="D: implied vs observed")
fig.show()


In [5]:
fig = plt.calibration_curve(C.probability_buckets(primary, "A"), title="A: implied vs observed")
fig.show()


## Interpretation: is there an unnormalised favourite/dog bias?
#
A common finding is that heavy favourites are overbet (their implied
probability exceeds realised frequency) while outsiders are underbet. We look
for any systematic bucket-level bias rather than claiming an edge.

In [6]:
for o in ("H", "A"):
    tbl = C.probability_buckets(primary, o)
    tbl = tbl[tbl["n"] >= 30]
    print(f"[{o}] mean implied vs observed by bucket (bias = observed - implied):")
    print(tbl[["bucket", "n", "implied", "observed", "diff"]].round(4).to_string())


[H] mean implied vs observed by bucket (bias = observed - implied):
      bucket    n  implied  observed    diff
0  0.10-0.20  179   0.1605    0.1620  0.0015
1  0.20-0.30  330   0.2568    0.2000 -0.0568
2  0.30-0.40  482   0.3547    0.3423 -0.0123
3  0.40-0.50  481   0.4519    0.4699  0.0179
4  0.50-0.60  373   0.5497    0.5764  0.0267
5  0.60-0.70  242   0.6519    0.7521  0.1002
6  0.70-0.80  141   0.7447    0.7801  0.0355
7  0.80-0.90   41   0.8299    0.8049 -0.0250
[A] mean implied vs observed by bucket (bias = observed - implied):
      bucket    n  implied  observed    diff
0  0.00-0.10  147   0.0789    0.0748 -0.0041
1  0.10-0.20  537   0.1544    0.1359 -0.0185
2  0.20-0.30  643   0.2503    0.2100 -0.0404
3  0.30-0.40  439   0.3473    0.3144 -0.0329
4  0.40-0.50  254   0.4454    0.5157  0.0704
5  0.50-0.60  150   0.5468    0.5600  0.0132
6  0.60-0.70   82   0.6411    0.6585  0.0175


## Calibration by season

In [7]:
for o in ("H", "D", "A"):
    bys = C.calibration_by_season(primary, o)
    g = bys.groupby("season").apply(
        lambda t: pd.Series({
            "mae": (t["diff"].abs() * t["n"]).sum() / t["n"].sum(),
            "bias": (t["diff"] * t["n"]).sum() / t["n"].sum(),
            "n": t["n"].sum(),
        }), include_groups=False)
    print(f"\n[{o}] calibration by season (MAE and bias weighted by n):")
    print(g.round(4).to_string())



[H] calibration by season (MAE and bias weighted by n):
           mae    bias      n
season                       
20/21   0.0624 -0.0031  367.0
21/22   0.0807 -0.0043  377.0
22/23   0.0536  0.0363  372.0
23/24   0.0824 -0.0107  375.0
24/25   0.0503 -0.0013  379.0
25/26   0.0654  0.0461  369.0

[D] calibration by season (MAE and bias weighted by n):
           mae    bias      n
season                       
20/21   0.0281  0.0238  377.0
21/22   0.0270  0.0270  380.0
22/23   0.0299 -0.0299  378.0
23/24   0.0315  0.0207  379.0
24/25   0.0122 -0.0068  380.0
25/26   0.0176 -0.0176  378.0

[A] calibration by season (MAE and bias weighted by n):
           mae    bias      n
season                       
20/21   0.0727 -0.0182  371.0
21/22   0.0415 -0.0163  380.0
22/23   0.0598 -0.0104  376.0
23/24   0.0460 -0.0096  380.0
24/25   0.0475  0.0080  372.0
25/26   0.0384 -0.0326  373.0


## Summary
#
Bet365's closing probabilities are reasonably calibrated. Implied and realised
frequencies generally move together; small residual biases exist but we treat
them as descriptive, not as exploitable edges. Any strategy claim must be
judged on unseen (out-of-sample) data.